# SECOM – Week 2: Data Preparation and Baseline

Armand Bussolino, Aurélien Roche, Jérémie Leroy-Dubief

This notebook prepares the SECOM data and evaluates two simple baselines. The test set is kept aside for the final evaluation.

### Load the data

Download `secom.data` and `secom_labels.data` from the [UCI SECOM dataset](https://archive.ics.uci.edu/dataset/179/secom). Put them in a `data/` folder at the root of the GitHub repository. This notebook should be in `notebooks/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, recall_score, precision_score,
                             f1_score, balanced_accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay)

# Try common folders (VS Code can start in different locations)
folders = [Path('data'), Path('../data'), Path('.'), Path('..')]
data_dir = next((p for p in folders if (p / 'secom.data').exists() and (p / 'secom_labels.data').exists()), None)
if data_dir is None:
    raise FileNotFoundError('Place secom.data and secom_labels.data in the data/ folder of your repository.')

X = pd.read_csv(data_dir / 'secom.data', sep=r'\s+', header=None)
labels = pd.read_csv(data_dir / 'secom_labels.data', sep=r'\s+', header=None)
X.columns = [f'sensor_{i}' for i in range(X.shape[1])]
y = labels.iloc[:, 0].astype(int)

print('X:', X.shape)
print('y:', y.shape)
print('Classes:', y.value_counts().to_dict())
X.head()

### Quick data checks

The dataset has many sensor columns and few failing samples. We first check missing values and constant columns, without modifying the original dataset.

In [ ]:
print('Missing values:', int(X.isna().sum().sum()))
print('Missing values (%):', round(X.isna().mean().mean() * 100, 2))
print('Columns with missing values:', int((X.isna().sum() > 0).sum()))
print('All-missing columns:', int(X.isna().all().sum()))
print('Constant or empty columns:', int((X.nunique(dropna=True) <= 1).sum()))
print('Duplicate rows:', int(X.duplicated().sum()))

class_counts = y.value_counts().sort_index()
print('Pass (-1):', class_counts.loc[-1])
print('Fail (1):', class_counts.loc[1])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(['Pass', 'Fail'], [class_counts.loc[-1], class_counts.loc[1]])
axes[0].set_ylabel('Number of samples')
axes[0].set_title('Target distribution')
X.isna().mean().mul(100).hist(bins=20, ax=axes[1])
axes[1].set_xlabel('Missing values per feature (%)')
axes[1].set_ylabel('Number of features')
axes[1].set_title('Missing data')
plt.tight_layout()
plt.show()

### Train / test split

We keep 20% of the samples for the final test. `stratify=y` keeps the same Pass/Fail proportions in both parts. No imputation or scaling is done before the split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print('Training samples:', len(X_train), '| Fail:', (y_train == 1).sum())
print('Test samples:', len(X_test), '| Fail:', (y_test == 1).sum())
print('Test set saved for Week 4: no model evaluation on X_test yet.')

### Preprocessing and baseline models

The logistic regression pipeline uses median imputation, removes constant columns and standardizes the remaining values. Each step is fitted again within each training fold of cross-validation, so validation data does not influence preprocessing.

B0 predicts the most common class. B1 is logistic regression with balanced class weights.

In [ ]:
dummy = DummyClassifier(strategy='prior')

logistic = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('variance', VarianceThreshold()),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
])

models = {'Dummy': dummy, 'Logistic Regression': logistic}

### Cross-validation

We use 5 stratified folds, repeated 3 times. The main metric is average precision (AP), also used as a summary of the precision-recall curve. We report recall and F1 for the Fail class too. Accuracy alone would be misleading here.

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
scoring = {
    'AP': 'average_precision',
    'Recall': 'recall',
    'Precision': 'precision',
    'F1': 'f1',
    'Balanced accuracy': 'balanced_accuracy'
}

results = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    row = {'Model': name}
    for metric in scoring:
        values = scores['test_' + metric]
        row[metric] = values.mean()
        row[metric + ' std'] = values.std()
    results.append(row)

results_df = pd.DataFrame(results).set_index('Model')
print('Mean scores across 15 validation folds:')
display(results_df.round(3))

### Confusion matrices (out-of-fold training predictions)

For a simple view of the errors, we make one prediction per training sample using 5-fold CV. This is **not** the held-out test result. The classifier threshold is still 0.5; threshold tuning will be done later.

In [ ]:
cv_single = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, model) in zip(axes, models.items()):
    pred = cross_val_predict(model, X_train, y_train, cv=cv_single, method='predict')
    cm = confusion_matrix(y_train, pred, labels=[-1, 1])
    ConfusionMatrixDisplay(cm, display_labels=['Pass', 'Fail']).plot(ax=ax, colorbar=False)
    ax.set_title(name + ' (CV)')
plt.tight_layout()
plt.show()

### First observations

Fill this in **after running the notebook**, using the values from the table above. Keep it short.

- The dataset is imbalanced: only about 6.6% of the samples are Fail.
- The Dummy model mainly predicts Pass, so it does not detect failures at the default threshold.
- **Logistic regression AP:** ... (CV mean). **Fail recall:** ... (CV mean).
- Compared with Dummy, logistic regression ...
- Next, we can try feature selection and other models, but the test set should stay untouched until the final evaluation.

### What is kept for Week 3

We will compare the baselines with other models and feature selection. We can also study the decision threshold and deal with class imbalance. Any resampling such as SMOTE must happen inside the CV training folds (using an `imblearn` pipeline).